# Notebook 1: Channel Exploration

This notebook explores the three channel models used in this project:
- **AWGN** – Additive White Gaussian Noise (static, line-of-sight)
- **Rayleigh** – Flat fading with log-normal shadowing (typical urban)
- **Jakes** – Time-correlated Rayleigh fading (mobility/Doppler)

We visualise SNR traces, BLER curves, and the distribution of optimal MCS across SNR ranges.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.channel.simulator import (
    ChannelSimulator, ChannelConfig,
    bler_model, effective_throughput,
    NUM_MCS, MCS_TABLE, MCS_SNR_THRESHOLD_AWGN
)

sns.set_theme(style='whitegrid', font_scale=1.1)
COLORS = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
print('Imports OK')

## 1. MCS Table Overview

In [ ]:
import pandas as pd

mcs_df = pd.DataFrame(MCS_TABLE,
    columns=['MCS Index', 'Modulation Order', 'Code Rate', 'Spectral Eff (bps/Hz)'])
mcs_df['Modulation'] = mcs_df['Modulation Order'].map(
    {2: 'QPSK', 4: '16-QAM', 6: '64-QAM', 8: '256-QAM'})
mcs_df['SNR Threshold (dB)'] = MCS_SNR_THRESHOLD_AWGN

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spectral efficiency
bars = axes[0].bar(mcs_df['MCS Index'], mcs_df['Spectral Eff (bps/Hz)'],
                   color=[COLORS[i//5 % 4] for i in range(NUM_MCS)], alpha=0.85)
axes[0].set_xlabel('MCS Index')
axes[0].set_ylabel('Spectral Efficiency (bps/Hz)')
axes[0].set_title('Spectral Efficiency per MCS')
for mod, color, label in [(0,5,COLORS[0],'QPSK'),(5,9,COLORS[1],'16-QAM'),
                           (9,14,COLORS[2],'64-QAM'),(14,19,COLORS[3],'256-QAM')]:
    pass  # legend handled below

# SNR thresholds
axes[1].plot(mcs_df['MCS Index'], mcs_df['SNR Threshold (dB)'],
             'o-', color='#4C72B0', linewidth=2, markersize=6)
axes[1].set_xlabel('MCS Index')
axes[1].set_ylabel('SNR Threshold (dB, AWGN)')
axes[1].set_title('Required SNR per MCS @ 10% BLER')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/plots/mcs_table_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(mcs_df.to_string(index=False))

## 2. BLER Waterfall Curves

In [ ]:
snr_range = np.linspace(-10, 30, 300)
mcs_to_plot = [0, 4, 8, 12, 15, 18]
labels = [f'MCS {m} ({MCS_TABLE[m][1]}QAM, R={MCS_TABLE[m][2]:.2f})'
          for m in mcs_to_plot]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ch_type, title in zip(axes, ['awgn', 'rayleigh'],
                               ['AWGN Channel', 'Rayleigh Fading Channel']):
    for mcs, label, color in zip(mcs_to_plot, labels,
                                  plt.cm.viridis(np.linspace(0.1, 0.9, len(mcs_to_plot)))):
        blers = [bler_model(snr, mcs, ch_type) for snr in snr_range]
        ax.semilogy(snr_range, blers, label=label, color=color, linewidth=1.8)
    ax.axhline(0.1, color='red', linestyle='--', linewidth=1, label='BLER target (10%)')
    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel('BLER')
    ax.set_title(f'BLER Waterfall Curves — {title}')
    ax.legend(fontsize=8, loc='lower left')
    ax.set_ylim(1e-4, 1)
    ax.set_xlim(-10, 30)
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/bler_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SNR Traces: AWGN vs Rayleigh vs Jakes

In [ ]:
N = 500
configs = [
    ('awgn',     '#4C72B0', 'AWGN'),
    ('rayleigh', '#55A868', 'Rayleigh Fading'),
    ('jakes',    '#C44E52', "Jake's Correlated Fading"),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (ch_type, color, title) in zip(axes, configs):
    cfg = ChannelConfig(channel_type=ch_type, snr_mean_db=15.0, snr_std_db=6.0, seed=42)
    sim = ChannelSimulator(cfg)
    snr_trace = sim.generate_snr_trace(N)
    
    ax.plot(snr_trace, color=color, linewidth=0.9, alpha=0.9)
    ax.axhline(15.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Mean SNR')
    ax.fill_between(range(N), snr_trace, 15.0,
                    where=(snr_trace > 15), alpha=0.15, color=color)
    ax.fill_between(range(N), snr_trace, 15.0,
                    where=(snr_trace < 15), alpha=0.15, color='gray')
    ax.set_ylabel('SNR (dB)')
    ax.set_title(f'{title}  (std={snr_trace.std():.2f} dB)')
    ax.legend(fontsize=9)
    ax.set_ylim(-5, 40)

axes[-1].set_xlabel('Frame Index')
plt.suptitle('Channel SNR Traces — Three Models', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/plots/snr_traces_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Optimal MCS Distribution vs SNR

In [ ]:
snr_vals = np.linspace(-8, 28, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ch_type in zip(axes, ['awgn', 'rayleigh']):
    optimal_mcs = [ChannelSimulator.optimal_mcs(snr, ch_type) for snr in snr_vals]
    
    # Color by modulation order
    mod_colors = {
        range(0,5): '#4C72B0',   # QPSK
        range(5,9): '#55A868',   # 16-QAM  
        range(9,14): '#C44E52',  # 64-QAM
        range(14,19): '#8172B2', # 256-QAM
    }
    colors = []
    for m in optimal_mcs:
        c = '#4C72B0'
        if m >= 14: c = '#8172B2'
        elif m >= 9: c = '#C44E52'
        elif m >= 5: c = '#55A868'
        colors.append(c)
    
    ax.scatter(snr_vals, optimal_mcs, c=colors, s=8, alpha=0.7)
    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel('Optimal MCS Index')
    ax.set_title(f'Oracle MCS vs SNR — {ch_type.upper()}')
    ax.set_yticks(range(0, NUM_MCS, 2))
    
    # Add modulation region labels
    for (lo, hi, label, color) in [
        (0, 4, 'QPSK', '#4C72B0'), (5, 8, '16-QAM', '#55A868'),
        (9, 13, '64-QAM', '#C44E52'), (14, 18, '256-QAM', '#8172B2')
    ]:
        ax.axhspan(lo - 0.4, hi + 0.4, alpha=0.08, color=color, label=label)
    ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('../results/plots/optimal_mcs_vs_snr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Throughput vs SNR: Which MCS wins?

In [ ]:
snr_vals = np.linspace(-8, 28, 300)
mcs_to_plot = [0, 3, 6, 9, 12, 15, 18]

fig, ax = plt.subplots(figsize=(12, 6))

for mcs in mcs_to_plot:
    tputs = [effective_throughput(snr, mcs, 'rayleigh') / 1e6 for snr in snr_vals]
    ax.plot(snr_vals, tputs,
            label=f'MCS {mcs} ({MCS_TABLE[mcs][1]}QAM)',
            linewidth=1.5, alpha=0.85)

# Oracle envelope
oracle_tputs = [
    effective_throughput(snr, ChannelSimulator.optimal_mcs(snr, 'rayleigh'), 'rayleigh') / 1e6
    for snr in snr_vals
]
ax.plot(snr_vals, oracle_tputs, 'k--', linewidth=2.5, label='Oracle (optimal MCS)')

ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Effective Throughput (Mbps)')
ax.set_title('Throughput vs SNR by MCS — Rayleigh Fading, 20 MHz')
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.set_ylim(0, 180)
ax.set_xlim(-8, 28)

plt.tight_layout()
plt.savefig('../results/plots/throughput_vs_snr.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key observations from channel exploration:

1. **Jake's fading** has the highest SNR variance (std ~10 dB), making it hardest for reactive link adapters
2. **BLER waterfalls** are steep (~3–5 dB per decade of BLER improvement), so MCS selection errors are costly
3. **Optimal MCS** increases roughly linearly with SNR, with clear modulation boundaries
4. **The oracle envelope** shows that using the wrong MCS by just 2 indices can cost 20–30% throughput in the mid-SNR range

These observations motivate the ML approach: a model that captures SNR trends can exploit the oracle envelope more fully than a CQI lookup.